# Feature Engineering — AirIQ
Day 3: Merge AQ + ERA5, build lag features, save feature matrix

In [17]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Only these 8 stations passed the 85% completeness threshold from EDA
RELIABLE_STATIONS = [
    "BTM Layout",
    "Hebbal",
    "Hombegowda Nagar",
    "Jayanagar 5th Block",
    "Jigani",
    "Kasturi Nagar",
    "Peenya",
    "Silk Board",
]

## Section 2: ERA5 Weather Feature Derivation

In [18]:
data_dir = os.path.join("..", "data")
csv_files = glob.glob(os.path.join(data_dir, "Raw_data_1Hr_2025_site_*.csv"))

all_dfs = []
for file_path in csv_files:
    filename = os.path.basename(file_path)
    match = re.search(r'site_\d+_(.*?)_Bengaluru', filename)
    station_name = match.group(1).replace('_', ' ') if match else filename
    
    if station_name not in RELIABLE_STATIONS:
        continue  # skip low-quality stations
    
    df = pd.read_csv(file_path, na_values=['NA'])
    df['station_name'] = station_name
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    all_dfs.append(df)

aq_df = pd.concat(all_dfs, ignore_index=True)
print(f"Loaded {aq_df['station_name'].nunique()} stations, {len(aq_df):,} rows")
print("Stations:", sorted(aq_df['station_name'].unique()))

Loaded 8 stations, 70,080 rows
Stations: ['BTM Layout', 'Hebbal', 'Hombegowda Nagar', 'Jayanagar 5th Block', 'Jigani', 'Kasturi Nagar', 'Peenya', 'Silk Board']


In [19]:
weather_path = os.path.join(data_dir, "raw", "bengaluru_weather_raw.csv")
era5 = pd.read_csv(weather_path)
era5['valid_time'] = pd.to_datetime(era5['valid_time'])

# --- Derive Wind Speed (m/s) ---
# ERA5 gives u10 (eastward) and v10 (northward) components at 10m
era5['wind_speed'] = np.sqrt(era5['u10']**2 + era5['v10']**2)

# --- Derive Wind Direction (degrees, meteorological: 0=North, 90=East) ---
era5['wind_dir'] = (np.degrees(np.arctan2(-era5['u10'], -era5['v10'])) + 360) % 360

# --- Convert Temperature: Kelvin → Celsius ---
era5['temp_c'] = era5['t2m'] - 273.15

# --- Derive Relative Humidity from dew point (d2m) and temperature (t2m) ---
# Magnus formula — standard meteorological approximation
d2m_c = era5['d2m'] - 273.15   # dew point in Celsius
era5['rel_humidity'] = 100 * (
    np.exp((17.625 * d2m_c) / (243.04 + d2m_c)) /
    np.exp((17.625 * era5['temp_c']) / (243.04 + era5['temp_c']))
)

# --- Keep only the derived columns + timestamp ---
era5_clean = era5[['valid_time', 'wind_speed', 'wind_dir', 'temp_c', 'rel_humidity']].copy()
era5_clean = era5_clean.rename(columns={'valid_time': 'Timestamp'})

print("ERA5 clean shape:", era5_clean.shape)
print(era5_clean.describe().round(2))

ERA5 clean shape: (8760, 5)
                 Timestamp  wind_speed  wind_dir   temp_c  rel_humidity
count                 8760     8760.00   8760.00  8760.00       8760.00
mean   2025-07-02 11:30:00        3.16    181.75    23.63         73.17
min    2025-01-01 00:00:00        0.05      0.06    13.11         17.93
25%    2025-04-02 05:45:00        2.08     88.60    21.42         62.98
50%    2025-07-02 11:30:00        2.87    217.65    23.17         78.05
75%    2025-10-01 17:15:00        3.99    266.60    25.79         87.64
max    2025-12-31 23:00:00        9.70    359.85    34.46         99.45
std                    NaN        1.48     95.21     3.56         17.93


## Section 3: Merge AQ + ERA5 by Timestamp

In [20]:
# Merge on Timestamp — each AQ row gets the ERA5 weather for that exact hour
# ERA5 has one row per hour, AQ has one row per station per hour
# After merge: each AQ row gets the city-wide weather for that hour
merged_df = aq_df.merge(era5_clean, on='Timestamp', how='left')

print(f"Before merge: {len(aq_df):,} rows")
print(f"After merge:  {len(merged_df):,} rows")
print(f"Weather merge success: {merged_df['wind_speed'].notna().mean()*100:.1f}% rows have weather data")

Before merge: 70,080 rows
After merge:  70,080 rows
Weather merge success: 100.0% rows have weather data


## Section 4: Time-Based Features

In [21]:
merged_df['hour']       = merged_df['Timestamp'].dt.hour
merged_df['day_of_week'] = merged_df['Timestamp'].dt.dayofweek   # 0=Monday, 6=Sunday
merged_df['month']      = merged_df['Timestamp'].dt.month
merged_df['is_weekend'] = (merged_df['day_of_week'] >= 5).astype(int)
merged_df['is_night']   = ((merged_df['hour'] >= 22) | (merged_df['hour'] <= 5)).astype(int)
merged_df['is_rush_hour'] = (
    ((merged_df['hour'] >= 7) & (merged_df['hour'] <= 10)) |
    ((merged_df['hour'] >= 17) & (merged_df['hour'] <= 20))
).astype(int)

print("Time features added:", ['hour','day_of_week','month','is_weekend','is_night','is_rush_hour'])
print(merged_df[['hour','is_weekend','is_rush_hour']].describe())

Time features added: ['hour', 'day_of_week', 'month', 'is_weekend', 'is_night', 'is_rush_hour']


               hour    is_weekend  is_rush_hour
count  70080.000000  70080.000000  70080.000000
mean      11.500000      0.284932      0.333333
std        6.922236      0.451385      0.471408
min        0.000000      0.000000      0.000000
25%        5.750000      0.000000      0.000000
50%       11.500000      0.000000      0.000000
75%       17.250000      1.000000      1.000000
max       23.000000      1.000000      1.000000


## Section 5: Lag and Rolling Features

In [22]:
# IMPORTANT: Sort by station and time before creating lags
# Otherwise lags bleed across stations and corrupt the model
merged_df = merged_df.sort_values(['station_name', 'Timestamp']).reset_index(drop=True)

# Create lag features per station (use groupby to prevent cross-station contamination)
for lag in [1, 3, 6, 12, 24]:
    col_name = f'pm25_lag_{lag}h'
    merged_df[col_name] = (
        merged_df.groupby('station_name')['PM2.5 (µg/m³)']
        .shift(lag)
    )

# Rolling features (window applied within each station group)
merged_df['pm25_roll_mean_3h'] = (
    merged_df.groupby('station_name')['PM2.5 (µg/m³)']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
merged_df['pm25_roll_mean_24h'] = (
    merged_df.groupby('station_name')['PM2.5 (µg/m³)']
    .transform(lambda x: x.shift(1).rolling(window=24, min_periods=6).mean())
)
merged_df['pm25_roll_std_3h'] = (
    merged_df.groupby('station_name')['PM2.5 (µg/m³)']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).std())
)

print("Lag features created:", [f'pm25_lag_{l}h' for l in [1,3,6,12,24]])
print("Rolling features: pm25_roll_mean_3h, pm25_roll_mean_24h, pm25_roll_std_3h")
print(f"\nNull counts in lag features:")
lag_cols = [f'pm25_lag_{l}h' for l in [1,3,6,12,24]]
print(merged_df[lag_cols].isnull().sum())

Lag features created: ['pm25_lag_1h', 'pm25_lag_3h', 'pm25_lag_6h', 'pm25_lag_12h', 'pm25_lag_24h']
Rolling features: pm25_roll_mean_3h, pm25_roll_mean_24h, pm25_roll_std_3h

Null counts in lag features:
pm25_lag_1h     5144
pm25_lag_3h     5160
pm25_lag_6h     5184
pm25_lag_12h    5232
pm25_lag_24h    5328
dtype: int64


## Section 6: Target Variable

In [23]:
# Target: PM2.5 one hour into the future
# We shift PM2.5 backward by 1 within each station group
merged_df['pm25_next_1h'] = (
    merged_df.groupby('station_name')['PM2.5 (µg/m³)']
    .shift(-1)
)

print(f"Target variable 'pm25_next_1h' created")
print(f"Non-null target values: {merged_df['pm25_next_1h'].notna().sum():,}")
print(f"Null target values (expected — last row of each station): {merged_df['pm25_next_1h'].isna().sum()}")
print(f"\nTarget stats:")
print(merged_df['pm25_next_1h'].describe().round(2))

Target variable 'pm25_next_1h' created
Non-null target values: 64,936
Null target values (expected — last row of each station): 5144

Target stats:
count    64936.00
mean        28.04
std         24.50
min          0.02
25%         15.77
50%         23.50
75%         34.97
max        985.07
Name: pm25_next_1h, dtype: float64


## Section 7: Final Feature Selection and Cleaning

In [24]:
# Define the exact feature columns for XGBoost
# These are the only columns the model will see
FEATURE_COLUMNS = [
    # AQ pollutants (current hour — as context features)
    'PM10 (µg/m³)',
    'NO2 (µg/m³)',
    'SO2 (µg/m³)',
    'CO (mg/m³)',
    'BP (mmHg)',         # barometric pressure — already in AQ files
    
    # PM2.5 lag features
    'pm25_lag_1h',
    'pm25_lag_3h',
    'pm25_lag_6h',
    'pm25_lag_12h',
    'pm25_lag_24h',
    
    # PM2.5 rolling features
    'pm25_roll_mean_3h',
    'pm25_roll_mean_24h',
    'pm25_roll_std_3h',
    
    # ERA5 derived weather features
    'wind_speed',
    'wind_dir',
    'temp_c',
    'rel_humidity',
    
    # Time features
    'hour',
    'day_of_week',
    'month',
    'is_weekend',
    'is_rush_hour',
]

TARGET_COLUMN = 'pm25_next_1h'
META_COLUMNS  = ['Timestamp', 'station_name']

# Build final dataset
final_df = merged_df[META_COLUMNS + FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

print(f"Before dropping NaN rows: {len(final_df):,}")
final_df = final_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])
print(f"After dropping NaN rows:  {len(final_df):,}")
print(f"Rows dropped: {len(merged_df) - len(final_df):,} ({100*(len(merged_df)-len(final_df))/len(merged_df):.1f}%)")
print(f"\nFinal shape: {final_df.shape}")
print(f"Features: {len(FEATURE_COLUMNS)}")

Before dropping NaN rows: 70,080
After dropping NaN rows:  40,945
Rows dropped: 29,135 (41.6%)

Final shape: (40945, 25)
Features: 22


## Section 8: Feature Correlation with Target

In [25]:
# Quick sanity check: are the lag features actually correlated with target?
# If pm25_lag_1h has near-zero correlation, something went wrong
corr = final_df[FEATURE_COLUMNS + [TARGET_COLUMN]].corr()[TARGET_COLUMN].drop(TARGET_COLUMN)
corr_sorted = corr.abs().sort_values(ascending=False)

print("Top 10 features by correlation with pm25_next_1h:")
print(corr_sorted.head(10).round(3))
print("\nBottom 5 features by correlation:")
print(corr_sorted.tail(5).round(3))

Top 10 features by correlation with pm25_next_1h:
pm25_roll_mean_24h    0.597
pm25_roll_mean_3h     0.559
pm25_lag_1h           0.542
pm25_lag_3h           0.462
pm25_lag_24h          0.425
PM10 (µg/m³)          0.409
pm25_lag_12h          0.387
pm25_lag_6h           0.381
wind_dir              0.310
CO (mg/m³)            0.167
Name: pm25_next_1h, dtype: float64

Bottom 5 features by correlation:
BP (mmHg)       0.068
is_rush_hour    0.039
hour            0.015
is_weekend      0.012
day_of_week     0.005
Name: pm25_next_1h, dtype: float64


## Section 9: Save Feature Matrix

In [26]:
# Create processed directory if it doesn't exist
processed_dir = os.path.join(data_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, "feature_matrix.csv")
final_df.to_csv(output_path, index=False)

# Verify save
saved = pd.read_csv(output_path)
print(f"✅ Feature matrix saved to: {output_path}")
print(f"   Shape: {saved.shape}")
print(f"   Size on disk: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")
print(f"\nFirst 3 rows:")
print(saved.head(3).to_string())

✅ Feature matrix saved to: ..\data\processed\feature_matrix.csv
   Shape: (40945, 25)
   Size on disk: 9.0 MB

First 3 rows:
             Timestamp station_name  PM10 (µg/m³)  NO2 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  BP (mmHg)  pm25_lag_1h  pm25_lag_3h  pm25_lag_6h  pm25_lag_12h  pm25_lag_24h  pm25_roll_mean_3h  pm25_roll_mean_24h  pm25_roll_std_3h  wind_speed   wind_dir   temp_c  rel_humidity  hour  day_of_week  month  is_weekend  is_rush_hour  pm25_next_1h
0  2025-01-02 00:00:00   BTM Layout        107.39        37.70        10.58        0.44     939.65        49.89        55.31        36.54         19.14         46.97          51.593333           39.076250          3.222458    3.046300  75.368477  18.3631     88.975775     0            3      1           0             0         51.28
1  2025-01-02 01:00:00   BTM Layout        114.00        37.17        10.49        0.44     939.80        53.92        49.58        49.17         19.96         49.86          51.130000           39.365833 